# Pears 9

### Imports

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# Append local path and import custom classes
sys.path.append(os.path.abspath('../scripts'))
from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER

### Data

In [ ]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### ENGINE

In [ ]:
print("⚙️ INITIATING WALK-FORWARD OOS PIPELINE...")

# 1. Create a pure Date column for grouping
df['Date'] = df.index.date
unique_days = df['Date'].unique()

train_days = 30 # Train on 1 month of data
out_of_sample_results = []

print(f"Total trading days available: {len(unique_days)}")
print(f"Rolling Train Window: {train_days} days. Starting loop...\n")

# 2. The Walk-Forward Loop
for i in range(train_days, len(unique_days)):
    
    # Define the chronological boundaries
    train_dates = unique_days[i - train_days : i]
    test_date = unique_days[i]
    
    train_df = df[df['Date'].isin(train_dates)].copy()
    test_df = df[df['Date'] == test_date].copy()
    
    # --- FIT ON THE PAST ---
    engine = ENGINE(train_df)
    train_fitted = engine.fit_cointegration(z_window=1000)
    engine.fit_markov_regimes()
    
    # --- PREDICT THE FUTURE ---
    oos_predictions = engine.predict_oos(test_df, train_fitted, z_window=1000)
    out_of_sample_results.append(oos_predictions)
    
    # Print progress every 10 days to ensure it hasn't stalled
    if i % 10 == 0:
        print(f"✅ Processed up to {test_date} | OOS Bars Generated: {len(oos_predictions)}")

# 3. Stitch the completely blind predictions together
live_trading_data = pd.concat(out_of_sample_results)
print(f"\n🎉 OOS Dataset Built: {len(live_trading_data)} totally blind trading bars ready for backtest.")

### BACKTEST

In [ ]:
print("🚀 RUNNING INSTITUTIONAL BACKTEST ON OOS DATA...")

# Feed the stitched live data into the backtester
bt = BACKTESTER(live_trading_data)

# Run the simulation
df_results = bt.run(
    entry_z=2.0,           # Enter trade at 2 standard deviations
    exit_z=0.0,            # Exit when spread crosses the mean
    danger_threshold=0.5,  # Flatten portfolio if HMM predicts >50% chance of Danger
    fee_bps=0.5            # Subtract 0.5 bps per trade for slippage/fees
)